# Module 04 — Lecture 2: HH GPU Solver (Euler & RK4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_04_hodgkin_huxley/02_hh_gpu_solver.ipynb)

---

The HH model presents richer challenges than LIF:
- 4 coupled ODEs per neuron (V, m, h, n) → heavier per-thread compute
- Small timestep (dt = 0.01 ms) → more kernel launches per millisecond
- Non-linear rate functions need efficient `__device__` implementations
- RK4 vs Euler trade-off: 4× more compute, ~4× larger timestep possible

**Learning objectives:**
- Implement `__device__` rate functions for voltage-dependent gating
- Write Euler and RK4 kernels for the 4-variable HH system
- Compare accuracy and throughput of the two solvers
- Visualise GPU-simulated action potentials

In [ ]:
!nvidia-smi

## 1. GPU Architecture of the HH Kernel

```
Thread i owns neuron i. Each step it:

1. Reads V[i], m[i], h[i], n[i], I[i] from global memory  (5 reads)
2. Computes alpha/beta rate functions  (in registers, ~30 FLOPs)
3. Computes ion currents  (~10 FLOPs)
4. Computes derivatives dV, dm, dh, dn  (~10 FLOPs)
5. Writes V[i], m[i], h[i], n[i] back  (4 writes)

Total: ~50 FLOPs / 9 global memory ops = 5.5 FLOPs/byte → compute-bound!
```

Unlike the LIF kernel (memory-bound), the HH kernel is **compute-bound**. The GPU spends more time calculating than waiting for memory. This means:
- Coalescing is less critical (we still do it, but it's not the bottleneck)
- Register usage matters more — keep all temporaries in registers
- The kernel benefits from GPU's high FP32 throughput directly

In [ ]:
%%writefile hh_solver.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e));exit(1);}} while(0)

__constant__ float c_Cm, c_gNa, c_gK, c_gL, c_ENa, c_EK, c_EL, c_dt;

// ── Device rate functions ─────────────────────────────────────────────────────
__device__ __forceinline__ float alpha_m(float V) {
    float dv = V + 40.0f;
    return (fabsf(dv) < 1e-5f) ? 1.0f : 0.1f * dv / (1.0f - expf(-dv/10.0f));
}
__device__ __forceinline__ float beta_m(float V)  { return 4.0f * expf(-(V+65.0f)/18.0f); }
__device__ __forceinline__ float alpha_h(float V) { return 0.07f * expf(-(V+65.0f)/20.0f); }
__device__ __forceinline__ float beta_h(float V)  { return 1.0f/(1.0f+expf(-(V+35.0f)/10.0f)); }
__device__ __forceinline__ float alpha_n(float V) {
    float dv = V + 55.0f;
    return (fabsf(dv) < 1e-5f) ? 0.1f : 0.01f * dv / (1.0f - expf(-dv/10.0f));
}
__device__ __forceinline__ float beta_n(float V)  { return 0.125f * expf(-(V+65.0f)/80.0f); }

// ── Compute derivatives (inlined for register efficiency) ─────────────────────
__device__ __forceinline__ void hh_deriv(
    float V, float m, float h, float n, float I,
    float* dV, float* dm, float* dh, float* dn
) {
    *dV = (I - c_gNa*m*m*m*h*(V-c_ENa) - c_gK*n*n*n*n*(V-c_EK) - c_gL*(V-c_EL)) / c_Cm;
    *dm = alpha_m(V)*(1.0f-m) - beta_m(V)*m;
    *dh = alpha_h(V)*(1.0f-h) - beta_h(V)*h;
    *dn = alpha_n(V)*(1.0f-n) - beta_n(V)*n;
}

// ── Euler kernel ──────────────────────────────────────────────────────────────
__global__ void hh_euler(float* V, float* m, float* h, float* n, const float* I, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    float dV, dm, dh, dn;
    hh_deriv(V[i], m[i], h[i], n[i], I[i], &dV, &dm, &dh, &dn);
    V[i] += c_dt*dV;
    m[i] = fmaxf(0.f, fminf(1.f, m[i] + c_dt*dm));
    h[i] = fmaxf(0.f, fminf(1.f, h[i] + c_dt*dh));
    n[i] = fmaxf(0.f, fminf(1.f, n[i] + c_dt*dn));
}

// ── RK4 kernel ────────────────────────────────────────────────────────────────
__global__ void hh_rk4(float* V, float* m, float* h, float* n, const float* I, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    float Vi=V[i], mi=m[i], hi=h[i], ni=n[i], Ii=I[i], dt=c_dt;
    float k1V,k1m,k1h,k1n, k2V,k2m,k2h,k2n, k3V,k3m,k3h,k3n, k4V,k4m,k4h,k4n;
    hh_deriv(Vi, mi, hi, ni, Ii, &k1V,&k1m,&k1h,&k1n);
    hh_deriv(Vi+.5f*dt*k1V, mi+.5f*dt*k1m, hi+.5f*dt*k1h, ni+.5f*dt*k1n, Ii, &k2V,&k2m,&k2h,&k2n);
    hh_deriv(Vi+.5f*dt*k2V, mi+.5f*dt*k2m, hi+.5f*dt*k2h, ni+.5f*dt*k2n, Ii, &k3V,&k3m,&k3h,&k3n);
    hh_deriv(Vi+   dt*k3V, mi+   dt*k3m, hi+   dt*k3h, ni+   dt*k3n, Ii, &k4V,&k4m,&k4h,&k4n);
    V[i] = Vi + dt/6.f*(k1V+2.f*k2V+2.f*k3V+k4V);
    m[i] = fmaxf(0.f,fminf(1.f, mi+dt/6.f*(k1m+2.f*k2m+2.f*k3m+k4m)));
    h[i] = fmaxf(0.f,fminf(1.f, hi+dt/6.f*(k1h+2.f*k2h+2.f*k3h+k4h)));
    n[i] = fmaxf(0.f,fminf(1.f, ni+dt/6.f*(k1n+2.f*k2n+2.f*k3n+k4n)));
}

__global__ void init_hh(float* V, float* m, float* h, float* n, float* I,
                         int N, float V0, float I_mean, float I_range) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    V[i] = V0;
    float am=alpha_m(V0),bm=beta_m(V0),ah=alpha_h(V0),bh=beta_h(V0),an=alpha_n(V0),bn=beta_n(V0);
    m[i] = am/(am+bm); h[i] = ah/(ah+bh); n[i] = an/(an+bn);
    I[i] = I_mean + I_range*((float)i/(N-1.f) - 0.5f);
}

int main() {
    const int N=1000; const float T_ms=100.f, dt=0.01f;
    int T_steps=(int)(T_ms/dt);
    float Cm=1.f,gNa=120.f,gK=36.f,gL=0.3f,ENa=50.f,EK=-77.f,EL=-54.4f;
    CUDA_CHECK(cudaMemcpyToSymbol(c_Cm,&Cm,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_gNa,&gNa,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_gK,&gK,4));  CUDA_CHECK(cudaMemcpyToSymbol(c_gL,&gL,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_ENa,&ENa,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_EK,&EK,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_EL,&EL,4));   CUDA_CHECK(cudaMemcpyToSymbol(c_dt,&dt,4));

    float *dV,*dm,*dh,*dn,*dI;
    size_t fb=N*4;
    CUDA_CHECK(cudaMalloc(&dV,fb));CUDA_CHECK(cudaMalloc(&dm,fb));
    CUDA_CHECK(cudaMalloc(&dh,fb));CUDA_CHECK(cudaMalloc(&dn,fb));
    CUDA_CHECK(cudaMalloc(&dI,fb));

    int thr=256,blk=(N+thr-1)/thr;

    // Benchmark Euler
    init_hh<<<blk,thr>>>(dV,dm,dh,dn,dI,N,-65.f,10.f,10.f);
    CUDA_CHECK(cudaDeviceSynchronize());
    cudaEvent_t t0,t1; float ms;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));
    for(int s=0;s<T_steps;s++) hh_euler<<<blk,thr>>>(dV,dm,dh,dn,dI,N);
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));
    printf("Euler: N=%d T=%.0fms dt=%.3fms  GPU=%.2fms  throughput=%.1fM steps/s\n",
           N,T_ms,dt,ms,(float)N*T_steps/ms/1e3);

    // Benchmark RK4
    init_hh<<<blk,thr>>>(dV,dm,dh,dn,dI,N,-65.f,10.f,10.f);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaEventRecord(t0));
    for(int s=0;s<T_steps;s++) hh_rk4<<<blk,thr>>>(dV,dm,dh,dn,dI,N);
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));
    printf("RK4:   N=%d T=%.0fms dt=%.3fms  GPU=%.2fms  throughput=%.1fM steps/s\n",
           N,T_ms,dt,ms,(float)N*T_steps/ms/1e3);

    // Save traces
    float* hV=(float*)malloc(N*4);
    CUDA_CHECK(cudaMemcpy(hV,dV,N*4,cudaMemcpyDeviceToHost));
    printf("Final V[0..4]: %.2f %.2f %.2f %.2f %.2f\n",
           hV[0],hV[1],hV[2],hV[3],hV[4]);
    free(hV);
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(dV);cudaFree(dm);cudaFree(dh);cudaFree(dn);cudaFree(dI);
    return 0;
}

In [ ]:
!nvcc -O2 -o hh_solver hh_solver.cu -lm && ./hh_solver

## 2. Euler vs RK4: Accuracy and Efficiency

| Solver | Error order | Steps for dt=0.1 ms | Relative cost |
|--------|-------------|---------------------|---------------|
| Euler | O(dt) | 10,000/ms | 1× |
| RK4 | O(dt⁴) | 2,500/ms (can use 4× larger dt) | 4× per step, 1× total |

For the HH model with its sharp Na⁺ spike (V changes >100 mV in 1 ms), **Euler with dt=0.01 ms** is generally sufficient. RK4 allows dt=0.05–0.1 ms with comparable accuracy, trading more compute per step for fewer steps.

**Rule of thumb:** Use Euler for large-scale network simulations (speed matters). Use RK4 when accurate single-neuron dynamics are the focus (e.g., computing accurate ISIs, comparing with experimental data).

In [ ]:
# Read voltage trace and visualise action potentials
# (Run hh_simulation from src/ first to generate hh_voltage.txt)
import subprocess, numpy as np, matplotlib.pyplot as plt

# Compile the full simulation
!nvcc -O2 -o hh_full ../src/hh_simulation.cu -lm 2>/dev/null || true
subprocess.run(['./hh_full', '100', '100', 'rk4'], capture_output=True)

try:
    data = np.loadtxt('hh_voltage.txt', comments='#')
    t    = data[:, 0]
    V_neurons = data[:, 1:]

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = plt.cm.viridis(np.linspace(0, 1, V_neurons.shape[1]))
    for j in range(min(10, V_neurons.shape[1])):
        ax.plot(t, V_neurons[:, j], color=colors[j], lw=0.8, alpha=0.9,
                label=f'Neuron {j} (I={5+j} μA/cm²)' if j < 5 else None)

    ax.set_xlabel('Time (ms)', fontsize=13)
    ax.set_ylabel('Membrane Voltage (mV)', fontsize=13)
    ax.set_title('HH Neurons — GPU Simulation (RK4, 10 of 100 neurons)\nColour = different I_ext values', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('hh_gpu_traces.png', dpi=150, bbox_inches='tight')
    plt.show()
except:
    print("Run hh_full first to generate hh_voltage.txt")

## Summary

| Feature | Implementation |
|---------|---------------|
| Rate functions | `__device__ __forceinline__` for zero-overhead calls |
| 4-variable ODE | Registers hold all temporaries — no shared memory needed |
| Euler | 1 derivative eval per step, O(dt) error |
| RK4 | 4 derivative evals per step, O(dt⁴) error |
| Memory layout | SoA: V[N], m[N], h[N], n[N] |

**Next lecture:** Parameter sweeps — simulate thousands of HH neurons with different I values simultaneously.

---

**Next →** [03 — Parameter Sweep](03_parameter_sweep.ipynb) &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_04_hodgkin_huxley/03_parameter_sweep.ipynb)